# Rapport - Méthode des volumes finis

# 1. Objectif
L’objectif de ce projet est de simuler le transport d’un constituant à travers un labyrinthe en utilisant un champ d’écoulement. Pour cela, on étudie l’évolution de la concentration au cours du temps et sa propagation dans le domaine.

![Labyrinthe étudié](laby.jpg)

Le transport du constituant est décrit par l’équation d’advection bidimensionnelle. Celle-ci exprime l’évolution de la concentration c sous l’effet du champ de vitesse, dont les composantes horizontale et verticale sont notées u et v :

$
\frac{\partial c}{\partial t}
+\frac{\partial (uc)}{\partial x}
+\frac{\partial (vc)}{\partial y}
=0
$

## 1.1 Simplification de l'équation d'advection
LL’équation d’advection est d’abord écrite sous sa forme conservative : les termes de transport apparaissent comme les dérivées des flux uc et vc. En appliquant la règle de dérivation d’un produit, on peut développer ces termes.

Dans notre problème, l’écoulement est supposé incompressible. La divergence du champ de vitesse est donc nulle :

$
\frac{\partial u}{\partial x}
+
\frac{\partial v}{\partial y}
=0
$

En appliquant la règle de dérivation d’un produit, on obtient :

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+c\frac{\partial u}{\partial x}
+v\frac{\partial c}{\partial y}
+c\frac{\partial v}{\partial y}
=0
$

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+v\frac{\partial c}{\partial y}
+c\underbrace{\left(
\frac{\partial u}{\partial x}
+\frac{\partial v}{\partial y}
\right)}_{=0}
=0
$

Puisque la divergence du champ de vitesse est nulle, le terme contenant c s’annule. L’équation se simplifie alors sous la forme non conservative :

$
\frac{\partial c}{\partial t}
+u\frac{\partial c}{\partial x}
+v\frac{\partial c}{\partial y}
=0
$

## 2.1. Lecture du labyrinthe et conversion en matrice
La première étape consiste à lire l’image du labyrinthe et à la convertir en une représentation numérique utilisable pour les calculs. Cette opération est réalisée par la fonction `compute()`, définie dans le fichier `laplace_only.py`.

In [4]:
def compute(filepath:str, fileout:str = 'out'):
    # Import du module PIL pour lire n'importe quelle image
    from PIL import Image
    # Lecture d'une image
    im = Image.open(filepath)

    # Conversion de l'image en matrice Numpy
    array = np.asarray(im)[:,:,0].copy() # uniquement la composante R d'une image RGB --> à particulariser si l'image est pure N&B par exemple

    # La matrice contient des valeurs comprises entre 0 et 255
    
    # Filtrage du 'gris' --> image N&B
    #  seuil arbitraire à 200
    array[np.where(array<=200)]= 0
    array[np.where(array>200)] = 1

    # Calcul du laplacien du potentiel
    domain, pot = laplace_pot(array,
                            outside_value=0,
                            inside_value=1,
                            pot_in=100.,
                            pot_out=-100.)

    fig, ax = plt.subplots(1,1)
    fig.set_size_inches(10,10)
    ax.imshow(pot)
    fig.savefig(fileout+'.png')

    ax.imshow(domain)
    fig.savefig('domain.png')

    np.savetxt(fileout+'_domain.txt', domain, fmt='%i')
    np.savetxt(fileout+'_pot.txt', pot, fmt='%15.10f')
    np.save(fileout+'_pot.npy', pot)


    return domain, pot

 L’image est d’abord ouverte à l’aide de la bibliothèque `PIL`, puis convertie en un tableau NumPy à deux dimensions. Chaque élément de ce tableau correspond à un pixel de l’image. L’indexation `[:, :, 0]` permet de sélectionner la première composante de couleur de chaque pixel, correspondant au rouge dans une image RGB. Les valeurs obtenues sont comprises entre 0 et 255, où 0 correspond au noir et 255 au blanc.

Afin de distinguer les murs des chemins du labyrinthe, un seuil de 200 est appliqué à cette matrice. Les pixels dont la composante rouge est inférieure ou égale à 200 reçoivent la valeur 0, tandis que ceux dont la composante rouge est strictement supérieure à 200 reçoivent la valeur 1. Cette opération, appelée binarisation, permet de réduire les différentes nuances de gris à deux valeurs distinctes. Le seuil de 200 constitue un choix arbitraire permettant de séparer les pixels sombres des pixels clairs.

La matrice binaire ainsi obtenue est ensuite transmise à la fonction `laplace_pot()`, qui sera présentée dans la section suivante. Celle-ci permet notamment de construire le domaine de calcul et de déterminer le potentiel de vitesse. 

## 2.2. Calcul du potentile de vitesse
Après la conversion de l’image en matrice binaire, la fonction `laplace_pot()` calcule le potentiel de vitesse dans le domaine défini.

La fonction reçoit notamment la matrice provisoire du domaine définie par `compute()`, la valeur 1 pour les cellules appartenant au domaine et la valeur 0 pour les cellules situées à l’extérieur. Elle reçoit également les valeurs imposées du potentiel aux limites : 100 à l’entrée et −100 à la sortie.

In [5]:
def laplace_pot(domain, outside_value = 0, inside_value = 1, pot_in = 1., pot_out = -1.):

_IncompleteInputError: incomplete input (838815357.py, line 1)

Elle commence par recadrer le domaine afin de supprimer les marges inutiles. Pour cela, elle détermine les indices $i_1$, $i_2$, $j_1$ et $j_2$, qui correspondent respectivement à la première et à la dernière ligne, puis à la première et à la dernière colonne retenues pour le recadrage.

In [ ]:
    outside= np.where(domain==outside_value)
    i1 = outside[0].min()
    i2 = outside[0].max()
    j1 = outside[1].min()
    j2 = outside[1].max()

    domain=domain[i1:i2+1,j1:j2+1]

Après le recadrage, la fonction identifie à nouveau les cellules situées à l’intérieur du domaine et celles qui se trouvent à l’extérieur. Cette étape est nécessaire, car les dimensions de la matrice ont changé, donc les coordonnées utilisées avant le recadrage ne correspondent plus nécessairement aux positions des cellules dans la nouvelle matrice.

In [ ]:
    outside= np.where(domain==outside_value)
    inside = np.where(domain==inside_value)

Elle compte ensuite les cellules appartenant au domaine. Chacune correspond à une inconnue du système linéaire, à savoir le potentiel recherché. Une matrice de numérotation est créée pour associer chaque cellule du domaine à une position dans le vecteur des inconnues x. Cette correspondance permet de construire le système Ax=b.

In [ ]:
    # Nombre de points de calcul
    nb = len(inside[0])

    # Matrice de numérotation
    numb_vec  = np.arange(0, nb, dtype=np.int32)
    #   Utilisation d'un stockage CSR par facilité --> mais "lent" pour la recherche des voisins (environ 60x plus lent pour l'image d'exemple)
    # numbering = csr((numb_vec, (inside[0], inside[1])), shape=array.shape)
    #
    # Utilisation d'une matrice pleine --> plus "rapide" pour la recherche des voisins mais demande éventuellement plus de mémoire
    numbering = np.zeros(domain.shape, dtype=np.int32)
    numbering[inside] = numb_vec

Les cellules d’entrée et de sortie sont ensuite repérées sur la première et la dernière ligne du domaine, afin d’identifier les emplacements où les valeurs du potentiel sont imposées.

In [ ]:
    # Recherche des zones CL
    #  - amont : ligne supérieure
    #  - aval : ligne inférieure
    inlet  = np.where(domain[0,:]==inside_value)[0]
    outlet = np.where(domain[-1,:]==inside_value)[0]

Le vecteur b est construit en tenant compte des conditions aux limites appliquées à l’entrée et à la sortie du domaine.

In [ ]:
    b  = np.zeros(nb)
    b[0:len(inlet)] = pot_in
    b[-len(outlet):]= pot_out

La matrice A du système Ax=b est construite en identifiant les relations entre les cellules du domaine et leurs voisines directes sous formes de listes. Pour chaque cellule intérieure, le code recherche les éventuels voisins situés en haut, en bas, à gauche et à droite. Lorsqu'un voisin appartient également au domaine, un coefficient égal à 1 est ajouté dans la matrice à la position correspondant aux deux cellules. Les cellules situées à l'entrée et à la sortie sont traitées séparément afin d'imposer les conditions aux limites associées aux potentiels fixés. Cette procédure permet de construire les coefficients du système d'équations linéaires qui sera ensuite résolu pour déterminer le potentiel dans le labyrinthe.

In [ ]:
# Remplissage de la matrice de poids en CSR = stockage compressé par lignes pour matrices creuses
    # *****************************************
    # Création par "List comprehension"
    Aij_inlet  = [ [1., numbering[0 ,j], numbering[0  ,j  ]] for j in list(inlet)]
    Aij_outlet = [ [1., numbering[-1,j], numbering[-1 ,j  ]] for j in list(outlet)]

    Aij_top    = [ [1., numbering[i ,j], numbering[i-1,j  ]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_top')    if i!=0 and i!=domain.shape[0]-1 if domain[i-1,j  ] == inside_value]
    Aij_bottom = [ [1., numbering[i ,j], numbering[i+1,j  ]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_bottom') if i!=0 and i!=domain.shape[0]-1 if domain[i+1,j  ] == inside_value]
    Aij_left   = [ [1., numbering[i ,j], numbering[i  ,j-1]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_left')   if i!=0 and i!=domain.shape[0]-1 if domain[i  ,j-1] == inside_value]
    Aij_right  = [ [1., numbering[i ,j], numbering[i  ,j+1]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_right')  if i!=0 and i!=domain.shape[0]-1 if domain[i  ,j+1] == inside_value]

    def count_neigh(i,j):

La fonction `count_neigh` compte le nombre de voisines d’une cellule qui appartiennent au domaine. 

In [ ]:
    def count_neigh(i,j):
        count = 0
        if domain[i-1,j] == inside_value:
            count+=1
        if domain[i+1,j] == inside_value:
            count+=1
        if domain[i,j-1] == inside_value:
            count+=1
        if domain[i,j+1] == inside_value:
            count+=1

        return float(count)

Pour chaque cellule intérieure qui n'est ni à la première ni à la dernière ligne, le code construit le coefficient associé à la celulle centrale égal à l'opposé du nombre de voisins appartenant au domaine.

In [ ]:
    Aij_center = [ [-count_neigh(i,j), numbering[i,j], numbering[i,j]] for i,j in tqdm(zip(inside[0], inside[1]), 'A_center') if i!=0 and i!=domain.shape[0]-1]

Les sept listes précédemment construites sont regroupées pour assembler la matrice creuse. La liste A contient les valeurs des coefficients, tandis que les listes ii et jj indiquent respectivement les indices de ligne et de colonne de chaque coefficient.

In [ ]:
    A_parts = [Aij_inlet, Aij_outlet, Aij_left, Aij_right, Aij_bottom, Aij_top, Aij_center]
    A  = [cur[0] for cur_part in A_parts for cur in cur_part]
    ii = [cur[1] for cur_part in A_parts for cur in cur_part]
    jj = [cur[2] for cur_part in A_parts for cur in cur_part]

    # Création de la CSR sur base des vecteurs de valeurs et des indices de position
    csr_a = csr((A,(ii,jj)))

Une fois la matrice creuse A, le vecteur des inconnues x et le vecteur b assemblés, le système linéaire

$
Ax=b
$

est résolu afin de déterminer les valeurs du potentiel.

In [ ]:
    # Résolution du système
    x = spsolve(csr_a,b)

Les valeurs du potentiel obtenues dans le vecteur x sont ensuite replacées dans une matrice ayant les mêmes dimensions que le domaine recadré. Cette représentation facilite l’interprétation et l’affichage du potentiel dans le labyrinthe.

In [ ]:
    pot = np.zeros(domain.shape, dtype=np.float64)
    pot[inside] = x

Une valeur est attribuée à chaque cellule selon son rôle dans le calcul : 0 pour les cellules situées hors du domaine, 1 pour les cellules appartenant au domaine et 2 pour les cellules d’entrée ou de sortie.

In [ ]:
    # Classification du domaine selon la convention
    #   - 0 = hors domaine
    #   - 1 = domaine intérieur
    #   - 2 = conditions limites
    domain[outside] = 0
    domain[inside]  = 1
    domain[0,inlet]   = 2
    domain[-1,outlet]  = 2

Enfin, une nouvelle matrice du domaine et une nouvelle matrice du potentiel sont créées en ajoutant deux lignes et deux colonnes de zéros aux bords. Ces marges permettent de disposer d’une bordure extérieure autour du domaine de calcul.

In [ ]:
    #ajout d'un bord de 0 sur tout le pourtour
    new_dom = np.zeros((domain.shape[0]+2, domain.shape[1]+2), dtype=np.int32)
    new_dom[1:-1,1:-1] = np.int32(domain)
    new_pot = np.zeros((domain.shape[0]+2, domain.shape[1]+2), dtype=np.float64)
    new_pot[1:-1,1:-1] = pot


La fonction renvoie finalement les matrices du domaine et du potentiel dans leur forme finale.

In [ ]:
    return new_dom, new_pot

<table>
  <tr>
    <td align="center">
      <img src="out.png" width="450">
      <br>
    </td>
    <td align="center">
      <img src="domain.png" width="550">
      <br>
    </td>
  </tr>
</table>

# 2.3. Calcul du champ de vitesse
La fonction `calculer_vitesses()`, définie dans le fichier vitesses.py, calcule les composantes horizontale u et verticale v du champ de vitesse à partir du potentiel.

In [ ]:
import numpy as np

def calculer_vitesses(potentiel, domaine, dx, dy):
    u = np.zeros_like(potentiel)
    v = np.zeros_like(potentiel)

    for i in range(1, potentiel.shape[0]-1):
        for j in range(1, potentiel.shape[1]-1):
            if domaine[i, j] == 1 or domaine[i, j] == 2:
                if domaine[i, j-1] != 0:
                    u[i, j] = -(potentiel[i, j] - potentiel[i, j-1]) / dx

    for i in range(1, potentiel.shape[0]-1):
        for j in range(1,potentiel.shape[1]-1):
            if domaine[i, j] == 1 or domaine[i, j] == 2:
                if domaine[i+1, j] != 0:
                    v[i, j] = (potentiel[i, j] - potentiel[i+1, j]) / dy

    return u, v

La fonction prend en arguments les matrices domaine et potentiel, ainsi que les pas de discrétisation spatiale dx et dy. Elle initialise les matrices u et v à zéro, avec les mêmes dimensions que la matrice du potentiel. Les composantes de vitesse ne sont ensuite calculées que pour les cellules du domaine qui satisfont les conditions définies dans le code. Les autres valeurs restent nulles.

Les boucles parcourent les indices intérieurs de la matrice, en laissant de côté les lignes et colonnes de bord ajoutées autour du domaine. La composante horizontale u est calculée par différences finis décentrée amont, c'est à dire à partir du potentiel de la cellule courante et de celui de sa voisine de gauche.

$
u_{i,j}=-\frac{\phi_{i,j}-\phi_{i,j-1}}{\Delta x}.
$

Selon la convention adoptée dans le code, une valeur positive de u correspond à une vitesse orientée vers la droite.

La composante verticale v est calculée par différences finis décentrée aval, c'est à dire à partir du potentiel de la cellule courante et de celui de sa voisine située en dessous.

$
v_{i,j}=\frac{\phi_{i,j}-\phi_{i+1,j}}{\Delta y}.
$

Dans le code, une valeur positive de v correspond à une vitesse orientée vers le bas, conformément à la convention d’indexation de l’image, où l’indice de ligne augmente vers le bas.

Que ce soit pour u ou v, lorsque la voisine est une cellule murale, la composante normale à cette face reste nulle. Cette condition vise à représenter l’imperméabilité des parois.

Les composantes u et v sont stockées dans deux matrices distinctes. Elles représentent les composantes normales de la vitesse aux faces de la cellule représentées comme ceci. 

![Positions de u et v](cellule.png)

Les champs associés aux composantes u et v sont représentés à l’aide de la fonction `afficher_champs_vitesse()`.

In [ ]:
import matplotlib.pyplot as plt

def afficher_champs_vitesse(u, v, domaine):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    ny, nx = u.shape

    X, Y = np.meshgrid(np.arange(nx), np.arange(ny))

    pas = 10

    # Facteur d'amplification visuelle des flèches
    amplification = 20

    # --------------------------------------------------
    # Champ de vitesse horizontal u
    # --------------------------------------------------
    Xu = X - 0.5
    Yu = Y

    axes[0].imshow(domaine, origin='upper')

    axes[0].quiver(
        Xu[::pas, ::pas],
        Yu[::pas, ::pas],
        amplification * u[::pas, ::pas],
        np.zeros_like(u[::pas, ::pas]),
        color='red',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.003,
        headwidth=5,
        headlength=7
    )

    axes[0].set_xticks(np.arange(-0.5, nx, 1), minor=True)
    axes[0].set_yticks(np.arange(-0.5, ny, 1), minor=True)
    axes[0].grid(
        which='minor',
        linestyle='-',
        linewidth=0.5,
        color='black',
        alpha=0.5
    )

    axes[0].set_title("Champ de la composante de vitesse horizontale u")
    axes[0].set_xlabel("indices colonnes")
    axes[0].set_ylabel("indices lignes")

    # --------------------------------------------------
    # Champ de vitesse vertical v
    # --------------------------------------------------
    Xv = X
    Yv = Y + 0.5

    axes[1].imshow(domaine, origin='upper')

    axes[1].quiver(
        Xv[::pas, ::pas],
        Yv[::pas, ::pas],
        np.zeros_like(v[::pas, ::pas]),
        amplification * v[::pas, ::pas],
        color='blue',
        angles='xy',
        scale_units='xy',
        scale=1,
        width=0.003,
        headwidth=5,
        headlength=7
    )

    axes[1].set_xticks(np.arange(-0.5, nx, 1), minor=True)
    axes[1].set_yticks(np.arange(-0.5, ny, 1), minor=True)
    axes[1].grid(
        which='minor',
        linestyle='-',
        linewidth=0.5,
        color='black',
        alpha=0.5
    )

    axes[1].set_title("Champ de la composante de vitesse verticale v")
    axes[1].set_xlabel("indices colonnes")
    axes[1].set_ylabel("indices lignes")

    plt.tight_layout()
    plt.savefig("champs_vitesse.png", dpi=300, bbox_inches="tight")
    plt.show()

<img src="champs_vitesse.png" alt="Champ de u et v" width="750">

## 2.4. Vérification de la divergence nulle
La fonction `verification_divergence()`, définie dans le fichier `divergence.py`,calcule la divergence du champ de vitese et vérifie qu'elle soit nulle.

In [ ]:
import numpy as np

def verification_divergence(u, v, domaine, dx, dy):

    divergence = np.zeros_like(u)

    for i in range(2, u.shape[0]-2):
        for j in range(1, u.shape[1]-1):
            if domaine[i, j] != 0:

                du_dx = (u[i, j+1] - u[i, j]) / dx
                dv_dy = (v[i, j] - v[i-1, j]) / dy
                divergence[i, j] = du_dx + dv_dy

    print("Divergence maximale :", np.max(np.abs(divergence)))

    return divergence

La fonction reçoit les composantes horizontale u et verticale v, la matrice du domaine ainsi que les pas de discrétisation ena rguments. Elle initialise la matrice de divergence à zéro, puis calcule la divergence uniquement pour les cellules retenues par les boucles et appartenant au domaine.

Le calcul utilise des différences finies cdntrées,pour approcher les dérivées spatiales :

$
\nabla\cdot\vec{u}
\approx \frac{u_{i,j+1}-u_{i,j}}{\Delta x}
+\frac{v_{i,j}-v_{i-1,j}}{\Delta y}.
$

La boucle sur les lignes commence à l’indice 2, car le calcul de la dérivée verticale utilise la valeur $v_{i−1,j}$​. Cela évite d’effectuer ce calcul sur la première ligne intérieure, où la valeur utilisée peut être affectée par la bordure ajoutée à la matrice. Les colonnes sont simplement parcourues de l’indice 1 jusqu’à l’avant-dernière colonne.

La divergence maximale en valeur absolue obtenue est de $3,13\cdot10^{-13},ce qui est suffisament proche de 0 pour conclure que le calcule du champ de vitesse est correcte mais montre que la méthode par différecnes finis n'est pas exacte. 

## 3. Méthode de transport
### 3.1. Reconstruction linéaire

La reconstruction constante consiste à supposer que la concentration est uniforme à l’intérieur de chaque cellule. La valeur de concentration associée à une cellule est donc utilisée pour représenter la concentration sur ses faces.

Lors du calcul du flux entre deux cellules voisines, la concentration retenue sur la face commune dépend du sens de l’écoulement. On utilise la concentration de la cellule située en amont, c’est-à-dire celle d’où provient l’écoulement. Cette approche, appelée reconstruction amont (upwind), permet de tenir compte du sens de propagation du constituant et contribue à la stabilité du schéma numérique.

### 3.2. Condition limite

Il est nécessaire de définir une condition limite à l’entrée du domaine. Dans notre problème, la concentration du constituant entrant est fixée à une valeur unitaire (c=1). Cette condition est qualifiée de faible, car la concentration est imposée uniquement sur la face d’entrée concernée, et non sur l’ensemble du volume de la cellule adjacente.

Dans notre implémentation, aucune condition ne doit être appliquée à la sortie, car la vitesse y est toujours positive.

### 3.3. Calcul des flux aux faces
Le calcul des flux est réalisé par la fonction `calcul_flux()`, définie dans le fichier `transport.py`.

In [ ]:
def calcul_flux(c, u, v, domaine):

    flux_u = np.zeros_like(u, dtype=float)
    flux_v = np.zeros_like(v, dtype=float)

    zone = np.s_[1:-1, 1:-1]

    interieur = (domaine[zone] == 1) | (domaine[zone] == 2)

    # =====================================================
    # 1. Flux horizontaux
    # =====================================================

    u_int = u[zone]

    c_gauche = c[1:-1, :-2]
    c_cellule = c[zone]

    concentration_face = np.where(u_int > 0, c_gauche, c_cellule)

    flux_horizontal = u_int * concentration_face

    flux_u[zone] = np.where(interieur, flux_horizontal, 0.0)

    # =====================================================
    # 2. Flux verticaux
    # =====================================================

    # Condition à l'entrée verticale, comme dans ton code actuel
    entree_verticale = ((domaine[:-2, 1:-1] == 0) & (domaine[1:-1, 1:-1] == 2))

    c_int = c[zone]
    c_int[entree_verticale] = 1.0

    v_int = v[zone]

    c_cellule = c[zone]
    c_bas = c[2:, 1:-1]

    concentration_face = np.where(v_int > 0, c_cellule, c_bas)

    flux_vertical = v_int * concentration_face

    # Flux nul en dehors du domaine de calcul
    flux_v[zone] = np.where(interieur, flux_vertical, 0.0)

    return flux_u, flux_v

La fonction reçoit la concentration c, les composantes de vitesse u et v, ainsi que la matrice du domaine. Elle renvoie deux matrices contenant respectivement les flux horizontaux et verticaux.

Elle initialise à zéro les matrices `flux_u` et `flux_v`, qui ont les mêmes dimensions que les matrices de vitesse `u` et `v`. Elles contiendront respectivement les flux horizontaux et verticaux. L'initialisation à zéro permet notamment de conserver un flux nul dans les zones où aucun flux ne doit être calculé.

Ensuite, l'objet de sélection `zone` est défini afin de sélectionner les cellules intérieures du domaine, en excluant le contour. La variable `interieur` est alors définie comme un tableau booléen dont ses éléments valent `True` si la cellule correspond à un chemin ou une entrée/sortie et `False` dans le cas contraire.

Le calcul des flux commence par récupérer les vitesses horizontales u et verticales v des cellules intérieures. La fonction extrait ensuite les concentrations des cellules considérées ainsi que celles situées à gauche et au dessus de ces dernières.

La concentration à utiliser pour calculer le flux sur chaque face est ensuite déterminée en fonction du signe de la vitesse horizontale/verticale :
- si la vitesse u est positive, la concentration retenue est celle située à gauche de la face;
- si la vitesse u est négative, la concentration retenue est celle de la cellule;
- si la vitesse v est positive, la concentration retenue est celle de la cellule;
- si la vitesse v est négative, la concentration retenue est celle de la cellule;

La concentration ainsi obtenue est ensuite multipliée par la vitesse pour calculer le flux :

$
F_{x/y} = u/v\,c
$

Ces flux sont ensuite enregistrés dans la partie intérieure de la matrice `flux_u`. L'instruction `np.where(interieur, flux_horizontal, 0.0)` permet de ne conserver ces flux que dans les cellules identifiées comme appartenant au chemin ou à une entrée/sortie. Les autres valeurs sont mises à zéro.

Le calcul des flux verticaux suit un principe similaire, à l'exception de l'imposition d'une condition limite. La fonction commence par identifier les cellules situées à l'entrée à l'aide d'un tableau booléens `entrées_verticales` qui repére les cellules dont la valeur dans `domaine` est égale à 2 et qui sont précédées par une cellule de mur. Pour ces cellules, la concentration est imposée à 1.